In [ ]:
import heapq
import random
from bisect import bisect_left
from collections import Counter, defaultdict, deque

# Trees, Tries and Heaps

---
# 1) Introduction

Tree questions are recursion questions. Almost every one fits this frame:

> *What does a node need from its children, and what does it return to its parent?*

| Style | When | Shape |
|---|---|---|
| **Bottom-up DFS** (post-order) | answer depends on subtrees: height, diameter, balanced, max path | `left = f(node.left); right = f(node.right); combine` |
| **Top-down DFS** (pre-order) | state flows from root: path sum, bounds for BST, max so far | `f(node, state)` passes updated state down |
| **BFS by level** | levels, right view, zigzag, minimum depth | `for _ in range(len(queue))` processes one level |
| **In-order on BST** | sorted order: k-th smallest, validate, successor | left → node → right visits keys in ascending order |

A **trie** is a tree keyed by characters, for prefix queries. A **heap** keeps the
min (or max) at the top in O(log n) per update — the tool for "top k", "k-th", streaming
medians and merging sorted sources. Python's `heapq` is a min-heap; push `-x` for a max-heap.

**How to use each question block.** Read the *Problem*, try for five minutes, then read the
*Signal*, then the *Idea*. Tests use LeetCode's level-order list format (`None` for a missing
child). Tiers: ★★★ Blind 75, ★★ NeetCode 150 / Grind 169 / Top Interview 150, ★ extended.

In [ ]:
class TreeNode:
    """Binary tree node."""

    def __init__(
        self,
        val: int = 0,
        left: "TreeNode | None" = None,
        right: "TreeNode | None" = None,
    ) -> None:
        self.val = val
        self.left = left
        self.right = right


def build_tree(values: list) -> TreeNode | None:
    """Tree from LeetCode level-order list with None gaps."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        for side in ("left", "right"):
            if i < len(values) and values[i] is not None:
                child = TreeNode(values[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root


def tree_to_list(root: TreeNode | None) -> list:
    """LeetCode level-order list of a tree (trailing None trimmed)."""
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node:
            out.append(node.val)
            queue.extend([node.left, node.right])
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out


def find(root: TreeNode | None, val: int) -> TreeNode | None:
    """First node with the given value (DFS)."""
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)


print(tree_to_list(build_tree([3, 9, 20, None, None, 15, 7])))

---
# 2) Binary Tree DFS

## 2.1 Bottom-up recursion

### 2.1.1 [104. Maximum Depth of Binary Tree](https://leetcode.com/problems/maximum-depth-of-binary-tree/) — ★★★ · Easy

**Problem.** Number of nodes on the longest root-to-leaf path.  
**Signal.** Node's answer = 1 + best child answer → bottom-up recursion.  
**Idea.** `depth(None) = 0`; `depth(node) = 1 + max(depth(left), depth(right))`.  
**Cost.** O(n) time · O(h) stack

In [ ]:
def max_depth(root: TreeNode | None) -> int:
    """Height of the tree in nodes."""
    if not root:
        return 0
    return 1 + max(max_depth(root.left), max_depth(root.right))


assert max_depth(build_tree([3, 9, 20, None, None, 15, 7])) == 3
assert max_depth(build_tree([1, None, 2])) == 2
assert max_depth(None) == 0
print(max_depth(build_tree([3, 9, 20, None, None, 15, 7])))

### 2.1.2 [111. Minimum Depth of Binary Tree](https://leetcode.com/problems/minimum-depth-of-binary-tree/) — ★ · Easy

**Problem.** Number of nodes on the shortest root-to-*leaf* path.  
**Signal.** Like max depth, but a missing child is not a leaf → BFS stops at the first leaf.  
**Idea.** BFS level by level; the first node with no children ends the search. (DFS must ignore a `None` child when the other child exists.)  
**Cost.** O(n) time · O(width) space

In [ ]:
def min_depth(root: TreeNode | None) -> int:
    """Depth of the shallowest leaf."""
    if not root:
        return 0
    queue = deque([(root, 1)])
    while queue:
        node, depth = queue.popleft()
        if not node.left and not node.right:
            return depth
        for child in (node.left, node.right):
            if child:
                queue.append((child, depth + 1))
    return 0


assert min_depth(build_tree([3, 9, 20, None, None, 15, 7])) == 2
assert min_depth(build_tree([2, None, 3, None, 4, None, 5, None, 6])) == 5
print(min_depth(build_tree([3, 9, 20, None, None, 15, 7])))

### 2.1.3 [100. Same Tree](https://leetcode.com/problems/same-tree/) — ★★★ · Easy

**Problem.** Are two trees structurally identical with equal values?  
**Signal.** Compare roots, then recurse on both pairs of children.  
**Idea.** Both empty → true; one empty or values differ → false; else both subtrees must match.  
**Cost.** O(n) time · O(h) stack

In [ ]:
def is_same_tree(p: TreeNode | None, q: TreeNode | None) -> bool:
    """True if trees are identical."""
    if not p or not q:
        return p is q
    return (
        p.val == q.val
        and is_same_tree(p.left, q.left)
        and is_same_tree(p.right, q.right)
    )


assert is_same_tree(build_tree([1, 2, 3]), build_tree([1, 2, 3]))
assert not is_same_tree(build_tree([1, 2]), build_tree([1, None, 2]))
print(is_same_tree(build_tree([1, 2, 1]), build_tree([1, 1, 2])))

### 2.1.4 [226. Invert Binary Tree](https://leetcode.com/problems/invert-binary-tree/) — ★★★ · Easy

**Problem.** Mirror the tree.  
**Signal.** Swap children at every node.  
**Idea.** Swap `left` and `right`, then invert both subtrees (any traversal order works).  
**Cost.** O(n) time · O(h) stack

In [ ]:
def invert_tree(root: TreeNode | None) -> TreeNode | None:
    """Mirror image of the tree."""
    if root:
        root.left, root.right = invert_tree(root.right), invert_tree(root.left)
    return root


assert tree_to_list(invert_tree(build_tree([4, 2, 7, 1, 3, 6, 9]))) == [
    4,
    7,
    2,
    9,
    6,
    3,
    1,
]
print(tree_to_list(invert_tree(build_tree([2, 1, 3]))))

### 2.1.5 [101. Symmetric Tree](https://leetcode.com/problems/symmetric-tree/) — ★★ · Easy

**Problem.** Is the tree a mirror of itself?  
**Signal.** Compare the left subtree with the mirrored right subtree.  
**Idea.** `mirror(a, b)`: both empty → true; values equal and `mirror(a.left, b.right)` and `mirror(a.right, b.left)`.  
**Cost.** O(n) time · O(h) stack

In [ ]:
def is_symmetric(root: TreeNode | None) -> bool:
    """True if the tree mirrors itself."""

    def mirror(a: TreeNode | None, b: TreeNode | None) -> bool:
        if not a or not b:
            return a is b
        return a.val == b.val and mirror(a.left, b.right) and mirror(a.right, b.left)

    return mirror(root.left, root.right) if root else True


assert is_symmetric(build_tree([1, 2, 2, 3, 4, 4, 3]))
assert not is_symmetric(build_tree([1, 2, 2, None, 3, None, 3]))
print(is_symmetric(build_tree([1, 2, 2, 3, 4, 4, 3])))

### 2.1.6 [572. Subtree of Another Tree](https://leetcode.com/problems/subtree-of-another-tree/) — ★★★ · Easy

**Problem.** Does `root` contain a subtree identical to `sub_root`?  
**Signal.** At each node of `root`, test Same Tree.  
**Idea.** `is_subtree(root) = same(root, sub) or is_subtree(root.left) or is_subtree(root.right)`. (Linear alternative: serialise both trees with null markers and do substring search.)  
**Cost.** O(m · n) time · O(h) stack

In [ ]:
def is_subtree(root: TreeNode | None, sub_root: TreeNode | None) -> bool:
    """True if sub_root appears as a subtree of root."""
    if not root:
        return False
    return (
        is_same_tree(root, sub_root)
        or is_subtree(root.left, sub_root)
        or is_subtree(root.right, sub_root)
    )


assert is_subtree(build_tree([3, 4, 5, 1, 2]), build_tree([4, 1, 2]))
assert not is_subtree(
    build_tree([3, 4, 5, 1, 2, None, None, None, None, 0]), build_tree([4, 1, 2])
)
print(is_subtree(build_tree([3, 4, 5, 1, 2]), build_tree([4, 1, 2])))

### 2.1.7 [543. Diameter of Binary Tree](https://leetcode.com/problems/diameter-of-binary-tree/) — ★★ · Easy

**Problem.** Longest path (in edges) between any two nodes.  
**Signal.** Return height upward, update a global best with `left + right` at each node.  
**Idea.** The longest path through a node uses the heights of both subtrees. The recursion returns height; a nonlocal variable records the best `left_h + right_h`.  
**Cost.** O(n) time · O(h) stack

In [ ]:
def diameter_of_binary_tree(root: TreeNode | None) -> int:
    """Longest path length in edges."""
    best = 0

    def height(node: TreeNode | None) -> int:
        nonlocal best
        if not node:
            return 0
        left, right = height(node.left), height(node.right)
        best = max(best, left + right)
        return 1 + max(left, right)

    height(root)
    return best


assert diameter_of_binary_tree(build_tree([1, 2, 3, 4, 5])) == 3
assert diameter_of_binary_tree(build_tree([1, 2])) == 1
print(diameter_of_binary_tree(build_tree([1, 2, 3, 4, 5])))

### 2.1.8 [110. Balanced Binary Tree](https://leetcode.com/problems/balanced-binary-tree/) — ★★ · Easy

**Problem.** Is the tree height-balanced (subtree heights differ by ≤ 1 everywhere)?  
**Signal.** Height recursion that also signals failure.  
**Idea.** Return the height, or −1 if a subtree is already unbalanced. One pass instead of recomputing heights at every node.  
**Cost.** O(n) time · O(h) stack

In [ ]:
def is_balanced(root: TreeNode | None) -> bool:
    """True if every node's subtrees differ in height by at most 1."""

    def height(node: TreeNode | None) -> int:
        if not node:
            return 0
        left, right = height(node.left), height(node.right)
        if left < 0 or right < 0 or abs(left - right) > 1:
            return -1
        return 1 + max(left, right)

    return height(root) >= 0


assert is_balanced(build_tree([3, 9, 20, None, None, 15, 7]))
assert not is_balanced(build_tree([1, 2, 2, 3, 3, None, None, 4, 4]))
print(is_balanced(build_tree([3, 9, 20, None, None, 15, 7])))

### 2.1.9 [124. Binary Tree Maximum Path Sum](https://leetcode.com/problems/binary-tree-maximum-path-sum/) — ★★★ · Hard

**Problem.** Maximum sum of any path (any start and end node, values may be negative).  
**Signal.** Diameter pattern with sums: return best downward chain, update global with both sides.  
**Idea.** `gain(node)` = node value + best non-negative child gain (negative branches are cut by `max(0, …)`). The best path *through* the node is `val + left_gain + right_gain`; only one branch may be returned upward.  
**Cost.** O(n) time · O(h) stack

In [ ]:
def max_path_sum(root: TreeNode | None) -> int:
    """Largest path sum anywhere in the tree."""
    best = float("-inf")

    def gain(node: TreeNode | None) -> int:
        nonlocal best
        if not node:
            return 0
        left, right = max(0, gain(node.left)), max(0, gain(node.right))
        best = max(best, node.val + left + right)
        return node.val + max(left, right)

    gain(root)
    return best


assert max_path_sum(build_tree([1, 2, 3])) == 6
assert max_path_sum(build_tree([-10, 9, 20, None, None, 15, 7])) == 42
assert max_path_sum(build_tree([-3])) == -3
print(max_path_sum(build_tree([-10, 9, 20, None, None, 15, 7])))

### 2.1.10 [236. Lowest Common Ancestor of a Binary Tree](https://leetcode.com/problems/lowest-common-ancestor-of-a-binary-tree/) — ★★ · Medium

**Problem.** Lowest node that has both `p` and `q` as descendants (a node is its own descendant).  
**Signal.** Bottom-up: return whichever target was found in each subtree.  
**Idea.** Return the node if it is `p` or `q`. Otherwise recurse; if both sides return non-null, this node splits `p` and `q` → it is the LCA. If only one side is non-null, pass it up.  
**Cost.** O(n) time · O(h) stack

In [ ]:
def lowest_common_ancestor(root, p: TreeNode, q: TreeNode) -> TreeNode | None:
    """LCA of p and q in a binary tree."""
    if not root or root is p or root is q:
        return root
    left = lowest_common_ancestor(root.left, p, q)
    right = lowest_common_ancestor(root.right, p, q)
    return root if left and right else left or right


t = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
assert lowest_common_ancestor(t, find(t, 5), find(t, 1)).val == 3
assert lowest_common_ancestor(t, find(t, 5), find(t, 4)).val == 5
print(lowest_common_ancestor(t, find(t, 6), find(t, 4)).val)

### 2.1.11 [1650. Lowest Common Ancestor of a Binary Tree III](https://leetcode.com/problems/lowest-common-ancestor-of-a-binary-tree-iii/) — ★ · Medium · Premium

**Problem.** Nodes have `parent` pointers (no root given); find the LCA of `p` and `q`.  
**Signal.** Two paths to the root that merge → same trick as Intersection of Two Linked Lists.  
**Idea.** Walk `a` from `p` and `b` from `q` up the parents; when one reaches the top, restart it at the other node. Both travel equal total distance and meet at the LCA.  
**Cost.** O(h) time · O(1) space

In [ ]:
def lca_with_parent(p, q):
    """LCA using parent pointers only."""
    a, b = p, q
    while a is not b:
        a = a.parent if a.parent else q
        b = b.parent if b.parent else p
    return a


t = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
stack = [(t, None)]
while stack:
    node, parent = stack.pop()
    node.parent = parent
    stack += [(c, node) for c in (node.left, node.right) if c]
assert lca_with_parent(find(t, 5), find(t, 1)).val == 3
assert lca_with_parent(find(t, 5), find(t, 4)).val == 5
print(lca_with_parent(find(t, 7), find(t, 8)).val)

### 2.1.12 [979. Distribute Coins in Binary Tree](https://leetcode.com/problems/distribute-coins-in-binary-tree/) — ★ · Medium

**Problem.** `n` nodes hold `n` coins in total; a move passes one coin along an edge. Moves needed so every node has exactly one coin?  
**Signal.** Each edge carries the surplus/deficit of the subtree below it.  
**Idea.** `excess(node) = coins + excess(left) + excess(right) − 1`. Every coin crossing the edge above a subtree counts once, so add `|excess(left)| + |excess(right)|` at every node.  
**Cost.** O(n) time · O(h) stack

In [ ]:
def distribute_coins(root: TreeNode | None) -> int:
    """Moves to give every node one coin."""
    moves = 0

    def excess(node: TreeNode | None) -> int:
        nonlocal moves
        if not node:
            return 0
        left, right = excess(node.left), excess(node.right)
        moves += abs(left) + abs(right)
        return node.val + left + right - 1

    excess(root)
    return moves


assert distribute_coins(build_tree([3, 0, 0])) == 2
assert distribute_coins(build_tree([0, 3, 0])) == 3
print(distribute_coins(build_tree([3, 0, 0])))

### 2.1.13 [337. House Robber III](https://leetcode.com/problems/house-robber-iii/) — ★ · Medium

**Problem.** Houses form a binary tree; robbing two directly linked houses is forbidden. Max loot?  
**Signal.** Tree DP: each node returns two values — (best if robbed, best if skipped).  
**Idea.** If a node is robbed, its children must be skipped: `val + left.skip + right.skip`. If skipped, each child independently picks its better state: `max(left) + max(right)`.  
**Cost.** O(n) time · O(h) stack

In [ ]:
def rob_tree(root: TreeNode | None) -> int:
    """Maximum loot with no two adjacent nodes robbed."""

    def dfs(node: TreeNode | None) -> tuple[int, int]:
        if not node:
            return 0, 0
        left, right = dfs(node.left), dfs(node.right)
        robbed = node.val + left[1] + right[1]
        skipped = max(left) + max(right)
        return robbed, skipped

    return max(dfs(root))


assert rob_tree(build_tree([3, 2, 3, None, 3, None, 1])) == 7
assert rob_tree(build_tree([3, 4, 5, 1, 3, None, 1])) == 9
print(rob_tree(build_tree([3, 4, 5, 1, 3, None, 1])))

### 2.1.14 [968. Binary Tree Cameras](https://leetcode.com/problems/binary-tree-cameras/) — ★ · Hard

**Problem.** A camera watches its node, parent and children. Minimum cameras to watch every node?  
**Signal.** Greedy bottom-up with three states per node.  
**Idea.** States: 0 = not covered, 1 = has camera, 2 = covered without camera. `None` counts as covered. If any child is uncovered, put a camera here. Else if any child has a camera, this node is covered. Else this node is uncovered and its parent will handle it. Placing cameras as high as possible (on parents of leaves) is optimal.  
**Cost.** O(n) time · O(h) stack

In [ ]:
def min_camera_cover(root: TreeNode | None) -> int:
    """Fewest cameras covering every node."""
    cameras = 0

    def state(node: TreeNode | None) -> int:
        nonlocal cameras
        if not node:
            return 2
        left, right = state(node.left), state(node.right)
        if left == 0 or right == 0:
            cameras += 1
            return 1
        if left == 1 or right == 1:
            return 2
        return 0

    if state(root) == 0:
        cameras += 1
    return cameras


assert min_camera_cover(build_tree([0, 0, None, 0, 0])) == 1
assert min_camera_cover(build_tree([0, 0, None, 0, None, 0, None, None, 0])) == 2
assert min_camera_cover(build_tree([0])) == 1
print(min_camera_cover(build_tree([0, 0, None, 0, 0])))

## 2.2 Top-down recursion and paths

### 2.2.1 [112. Path Sum](https://leetcode.com/problems/path-sum/) — ★★ · Easy

**Problem.** Is there a root-to-leaf path summing to `target`?  
**Signal.** State flows downward: remaining sum.  
**Idea.** Subtract the node value; at a leaf, check the remainder is 0.  
**Cost.** O(n) time · O(h) stack

In [ ]:
def has_path_sum(root: TreeNode | None, target: int) -> bool:
    """True if some root-to-leaf path sums to target."""
    if not root:
        return False
    target -= root.val
    if not root.left and not root.right:
        return target == 0
    return has_path_sum(root.left, target) or has_path_sum(root.right, target)


t = build_tree([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, None, 1])
assert has_path_sum(t, 22)
assert not has_path_sum(build_tree([1, 2, 3]), 5)
print(has_path_sum(t, 22))

### 2.2.2 [113. Path Sum II](https://leetcode.com/problems/path-sum-ii/) — ★★ · Medium

**Problem.** All root-to-leaf paths summing to `target`.  
**Signal.** Top-down DFS with a path list and backtracking.  
**Idea.** Append the node, recurse, pop. At a leaf with the right sum, copy the path.  
**Cost.** O(n · h) time (copying paths) · O(h) stack

In [ ]:
def path_sum_ii(root: TreeNode | None, target: int) -> list[list[int]]:
    """Every root-to-leaf path with the given sum."""
    out: list[list[int]] = []
    path: list[int] = []

    def dfs(node: TreeNode | None, remaining: int) -> None:
        if not node:
            return
        path.append(node.val)
        remaining -= node.val
        if not node.left and not node.right and remaining == 0:
            out.append(path.copy())
        dfs(node.left, remaining)
        dfs(node.right, remaining)
        path.pop()

    dfs(root, target)
    return out


t = build_tree([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, 5, 1])
assert path_sum_ii(t, 22) == [[5, 4, 11, 2], [5, 8, 4, 5]]
print(path_sum_ii(t, 22))

### 2.2.3 [437. Path Sum III](https://leetcode.com/problems/path-sum-iii/) — ★★ · Medium

**Problem.** Count downward paths (any start, any end) with sum `target`.  
**Signal.** Subarray Sum Equals K on every root-to-node path → prefix sums in a hash map.  
**Idea.** Carry the running prefix sum down. At each node, paths ending here number `count[prefix − target]`. Add the prefix before recursing into children and remove it after (backtracking keeps the map restricted to the current root path).  
**Cost.** O(n) time · O(h) space

In [ ]:
def path_sum_iii(root: TreeNode | None, target: int) -> int:
    """Number of downward paths summing to target."""
    count = Counter({0: 1})

    def dfs(node: TreeNode | None, prefix: int) -> int:
        if not node:
            return 0
        prefix += node.val
        found = count[prefix - target]
        count[prefix] += 1
        found += dfs(node.left, prefix) + dfs(node.right, prefix)
        count[prefix] -= 1
        return found

    return dfs(root, 0)


assert path_sum_iii(build_tree([10, 5, -3, 3, 2, None, 11, 3, -2, None, 1]), 8) == 3
t = build_tree([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, 5, 1])
assert path_sum_iii(t, 22) == 3
print(path_sum_iii(t, 22))

### 2.2.4 [129. Sum Root to Leaf Numbers](https://leetcode.com/problems/sum-root-to-leaf-numbers/) — ★★ · Medium

**Problem.** Each root-to-leaf path forms a number (digits); return the total.  
**Signal.** Top-down: pass the number built so far.  
**Idea.** `value = value * 10 + node.val`; at a leaf return it; otherwise sum both children.  
**Cost.** O(n) time · O(h) stack

In [ ]:
def sum_numbers(root: TreeNode | None) -> int:
    """Sum of numbers formed by root-to-leaf paths."""

    def dfs(node: TreeNode | None, value: int) -> int:
        if not node:
            return 0
        value = value * 10 + node.val
        if not node.left and not node.right:
            return value
        return dfs(node.left, value) + dfs(node.right, value)

    return dfs(root, 0)


assert sum_numbers(build_tree([1, 2, 3])) == 25
assert sum_numbers(build_tree([4, 9, 0, 5, 1])) == 1026
print(sum_numbers(build_tree([4, 9, 0, 5, 1])))

### 2.2.5 [1448. Count Good Nodes in Binary Tree](https://leetcode.com/problems/count-good-nodes-in-binary-tree/) — ★★ · Medium

**Problem.** A node is good if no node on the path from the root to it has a greater value. Count them.  
**Signal.** Top-down: pass the max seen so far.  
**Idea.** A node is good iff `val >= path_max`; recurse with `max(path_max, val)`.  
**Cost.** O(n) time · O(h) stack

In [ ]:
def good_nodes(root: TreeNode | None) -> int:
    """Number of nodes not smaller than any ancestor."""

    def dfs(node: TreeNode | None, path_max: int) -> int:
        if not node:
            return 0
        good = node.val >= path_max
        path_max = max(path_max, node.val)
        return good + dfs(node.left, path_max) + dfs(node.right, path_max)

    return dfs(root, root.val)


assert good_nodes(build_tree([3, 1, 4, 3, None, 1, 5])) == 4
assert good_nodes(build_tree([3, 3, None, 4, 2])) == 3
print(good_nodes(build_tree([3, 1, 4, 3, None, 1, 5])))

### 2.2.6 [257. Binary Tree Paths](https://leetcode.com/problems/binary-tree-paths/) — ★ · Easy

**Problem.** All root-to-leaf paths as strings `"1->2->5"`.  
**Signal.** Top-down DFS carrying the path.  
**Idea.** Pass the path string down; at a leaf, record it.  
**Cost.** O(n · h) time

In [ ]:
def binary_tree_paths(root: TreeNode | None) -> list[str]:
    """Root-to-leaf paths as arrow strings."""
    out: list[str] = []

    def dfs(node: TreeNode | None, path: str) -> None:
        if not node:
            return
        path += str(node.val)
        if not node.left and not node.right:
            out.append(path)
        dfs(node.left, path + "->")
        dfs(node.right, path + "->")

    dfs(root, "")
    return out


assert binary_tree_paths(build_tree([1, 2, 3, None, 5])) == ["1->2->5", "1->3"]
print(binary_tree_paths(build_tree([1, 2, 3, None, 5])))

### 2.2.7 [1372. Longest ZigZag Path in a Binary Tree](https://leetcode.com/problems/longest-zigzag-path-in-a-binary-tree/) — ★ · Medium

**Problem.** Longest path that alternates left/right child moves (counted in edges).  
**Signal.** Top-down state: (direction of last move, current length).  
**Idea.** Going left from a node that was reached by a right move extends the zigzag (`length + 1`); going the same direction again restarts at 1.  
**Cost.** O(n) time · O(h) stack

In [ ]:
def longest_zig_zag(root: TreeNode | None) -> int:
    """Longest alternating downward path length."""
    best = 0

    def dfs(node: TreeNode | None, went_left: bool, length: int) -> None:
        nonlocal best
        if not node:
            return
        best = max(best, length)
        dfs(node.left, True, 1 if went_left else length + 1)
        dfs(node.right, False, length + 1 if went_left else 1)

    dfs(root.left, True, 1)
    dfs(root.right, False, 1)
    return best


t = build_tree([1, None, 1, 1, 1, None, None, 1, 1, None, 1, None, None, None, 1])
assert longest_zig_zag(t) == 3
assert longest_zig_zag(build_tree([1, 1, 1, None, 1, None, None, 1, 1, None, 1])) == 4
assert longest_zig_zag(build_tree([1])) == 0
print(longest_zig_zag(t))

### 2.2.8 [872. Leaf-Similar Trees](https://leetcode.com/problems/leaf-similar-trees/) — ★ · Easy

**Problem.** Do two trees have the same leaf value sequence (left to right)?  
**Signal.** Collect leaves with DFS and compare.  
**Idea.** A generator yields leaves in order; compare the two sequences.  
**Cost.** O(n) time · O(h) stack

In [ ]:
def leaf_similar(a: TreeNode | None, b: TreeNode | None) -> bool:
    """True if both trees have the same leaf sequence."""

    def leaves(node: TreeNode | None):
        if node:
            if not node.left and not node.right:
                yield node.val
            yield from leaves(node.left)
            yield from leaves(node.right)

    return list(leaves(a)) == list(leaves(b))


a = build_tree([3, 5, 1, 6, 2, 9, 8, None, None, 7, 4])
b = build_tree([3, 5, 1, 6, 7, 4, 2, None, None, None, None, None, None, 9, 8])
assert leaf_similar(a, b)
assert not leaf_similar(build_tree([1, 2, 3]), build_tree([1, 3, 2]))
print(leaf_similar(a, b))

## 2.3 Construction, transformation and serialisation

### 2.3.1 [105. Construct Binary Tree from Preorder and Inorder Traversal](https://leetcode.com/problems/construct-binary-tree-from-preorder-and-inorder-traversal/) — ★★★ · Medium

**Problem.** Rebuild the tree from its preorder and inorder sequences (values distinct).  
**Signal.** Preorder gives the root; inorder splits left and right subtrees.  
**Idea.** The next preorder value is the current root. Its inorder index `m` means everything left of `m` is the left subtree. Recurse on inorder ranges, consuming preorder left to right; a value → index map makes each lookup O(1).  
**Cost.** O(n) time · O(n) space

In [ ]:
def build_from_pre_in(preorder: list[int], inorder: list[int]) -> TreeNode | None:
    """Tree with the given preorder and inorder traversals."""
    index = {v: i for i, v in enumerate(inorder)}
    pre = iter(preorder)

    def make(lo: int, hi: int) -> TreeNode | None:
        if lo > hi:
            return None
        root = TreeNode(next(pre))
        m = index[root.val]
        root.left = make(lo, m - 1)
        root.right = make(m + 1, hi)
        return root

    return make(0, len(inorder) - 1)


t = build_from_pre_in([3, 9, 20, 15, 7], [9, 3, 15, 20, 7])
assert tree_to_list(t) == [3, 9, 20, None, None, 15, 7]
print(tree_to_list(t))

### 2.3.2 [106. Construct Binary Tree from Inorder and Postorder Traversal](https://leetcode.com/problems/construct-binary-tree-from-inorder-and-postorder-traversal/) — ★★ · Medium

**Problem.** Rebuild the tree from inorder and postorder sequences.  
**Signal.** Postorder read backwards is root → right → left.  
**Idea.** Same as the preorder version, but consume postorder from the end and build the *right* subtree before the left.  
**Cost.** O(n) time · O(n) space

In [ ]:
def build_from_in_post(inorder: list[int], postorder: list[int]) -> TreeNode | None:
    """Tree with the given inorder and postorder traversals."""
    index = {v: i for i, v in enumerate(inorder)}
    post = reversed(postorder)

    def make(lo: int, hi: int) -> TreeNode | None:
        if lo > hi:
            return None
        root = TreeNode(next(post))
        m = index[root.val]
        root.right = make(m + 1, hi)
        root.left = make(lo, m - 1)
        return root

    return make(0, len(inorder) - 1)


t = build_from_in_post([9, 3, 15, 20, 7], [9, 15, 7, 20, 3])
assert tree_to_list(t) == [3, 9, 20, None, None, 15, 7]
print(tree_to_list(t))

### 2.3.3 [114. Flatten Binary Tree to Linked List](https://leetcode.com/problems/flatten-binary-tree-to-linked-list/) — ★★ · Medium

**Problem.** Flatten in place into a right-skewed "list" in preorder.  
**Signal.** Morris-style: splice each left subtree between the node and its right subtree.  
**Idea.** At node `cur` with a left child, find the rightmost node of the left subtree; attach `cur.right` there, move the left subtree to the right, clear left, step right.  
**Cost.** O(n) time · O(1) space

In [ ]:
def flatten(root: TreeNode | None) -> None:
    """Flatten tree to a preorder right-chain in place."""
    cur = root
    while cur:
        if cur.left:
            tail = cur.left
            while tail.right:
                tail = tail.right
            tail.right = cur.right
            cur.right, cur.left = cur.left, None
        cur = cur.right


t = build_tree([1, 2, 5, 3, 4, None, 6])
flatten(t)
assert tree_to_list(t) == [1, None, 2, None, 3, None, 4, None, 5, None, 6]
print(tree_to_list(t))

### 2.3.4 [617. Merge Two Binary Trees](https://leetcode.com/problems/merge-two-binary-trees/) — ★ · Easy

**Problem.** Overlay two trees: overlapping nodes sum, otherwise take the existing node.  
**Signal.** Simultaneous recursion on both trees.  
**Idea.** If one side is empty return the other; else make a node with the summed value and merge children pairwise.  
**Cost.** O(min(m, n)) time

In [ ]:
def merge_trees(a: TreeNode | None, b: TreeNode | None) -> TreeNode | None:
    """Overlay of two trees."""
    if not a or not b:
        return a or b
    return TreeNode(
        a.val + b.val, merge_trees(a.left, b.left), merge_trees(a.right, b.right)
    )


t = merge_trees(build_tree([1, 3, 2, 5]), build_tree([2, 1, 3, None, 4, None, 7]))
assert tree_to_list(t) == [3, 4, 5, 5, 4, None, 7]
print(tree_to_list(t))

### 2.3.5 [297. Serialize and Deserialize Binary Tree](https://leetcode.com/problems/serialize-and-deserialize-binary-tree/) — ★★★ · Hard

**Problem.** Encode a tree to a string and decode it back.  
**Signal.** Preorder with explicit null markers is unambiguous.  
**Idea.** Serialise with preorder, writing `#` for `None`. Deserialise by reading tokens in the same order with an iterator: a `#` returns `None`; otherwise create the node, then build its left and right subtrees recursively.  
**Cost.** O(n) time and space

In [ ]:
def serialize(root: TreeNode | None) -> str:
    """Preorder string with # for null."""
    out: list[str] = []

    def dfs(node: TreeNode | None) -> None:
        if not node:
            out.append("#")
            return
        out.append(str(node.val))
        dfs(node.left)
        dfs(node.right)

    dfs(root)
    return ",".join(out)


def deserialize(data: str) -> TreeNode | None:
    """Inverse of serialize."""
    tokens = iter(data.split(","))

    def make() -> TreeNode | None:
        tok = next(tokens)
        if tok == "#":
            return None
        return TreeNode(int(tok), make(), make())

    return make()


t = build_tree([1, 2, 3, None, None, 4, 5])
assert tree_to_list(deserialize(serialize(t))) == [1, 2, 3, None, None, 4, 5]
assert deserialize(serialize(None)) is None
print(serialize(t))

### 2.3.6 [863. All Nodes Distance K in Binary Tree](https://leetcode.com/problems/all-nodes-distance-k-in-binary-tree/) — ★★ · Medium

**Problem.** Values of all nodes at distance `k` from a `target` node.  
**Signal.** Distance can go upward → turn the tree into an undirected graph, then BFS.  
**Idea.** Record each node's parent with DFS. BFS from `target` over left, right and parent edges with a visited set; the frontier after `k` levels is the answer.  
**Cost.** O(n) time · O(n) space

In [ ]:
def distance_k(root: TreeNode, target: TreeNode, k: int) -> list[int]:
    """Values at exactly k edges from target."""
    parent: dict[TreeNode, TreeNode | None] = {root: None}
    stack = [root]
    while stack:
        node = stack.pop()
        for child in (node.left, node.right):
            if child:
                parent[child] = node
                stack.append(child)
    frontier, seen = [target], {target}
    for _ in range(k):
        nxt = []
        for node in frontier:
            for nb in (node.left, node.right, parent[node]):
                if nb and nb not in seen:
                    seen.add(nb)
                    nxt.append(nb)
        frontier = nxt
    return sorted(node.val for node in frontier)


t = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
assert distance_k(t, find(t, 5), 2) == [1, 4, 7]
single = build_tree([1])
assert distance_k(single, single, 3) == []
print(distance_k(t, find(t, 5), 2))

### 2.3.7 [222. Count Complete Tree Nodes](https://leetcode.com/problems/count-complete-tree-nodes/) — ★★ · Easy

**Problem.** Count nodes of a *complete* tree in less than O(n).  
**Signal.** If leftmost and rightmost depths match, the subtree is perfect: `2^h − 1` nodes.  
**Idea.** Compute the depth along the far left and far right. Equal → perfect, return the formula. Otherwise recurse on both children; one of them is always perfect, so only one branch goes deep.  
**Cost.** O(log² n) time

In [ ]:
def count_nodes(root: TreeNode | None) -> int:
    """Node count of a complete binary tree."""
    if not root:
        return 0
    left_h = right_h = 0
    node = root
    while node:
        left_h, node = left_h + 1, node.left
    node = root
    while node:
        right_h, node = right_h + 1, node.right
    if left_h == right_h:
        return 2**left_h - 1
    return 1 + count_nodes(root.left) + count_nodes(root.right)


assert count_nodes(build_tree([1, 2, 3, 4, 5, 6])) == 6
assert count_nodes(build_tree(list(range(1, 16)))) == 15
assert count_nodes(None) == 0
print(count_nodes(build_tree([1, 2, 3, 4, 5, 6])))

---
# 3) Binary Tree BFS

```python
queue = deque([root])
while queue:
    level = []
    for _ in range(len(queue)):        # exactly the nodes of the current level
        node = queue.popleft()
        level.append(node.val)
        queue.extend(c for c in (node.left, node.right) if c)
```

## 3.1 Level-order traversal

### 3.1.1 [102. Binary Tree Level Order Traversal](https://leetcode.com/problems/binary-tree-level-order-traversal/) — ★★★ · Medium

**Problem.** Values level by level, left to right.  
**Signal.** Level by level → BFS with the level-size loop.  
**Idea.** Snapshot `len(queue)` before processing; exactly that many nodes form the level.  
**Cost.** O(n) time · O(width) space

In [ ]:
def level_order(root: TreeNode | None) -> list[list[int]]:
    """Node values grouped by level."""
    out: list[list[int]] = []
    queue = deque([root] if root else [])
    while queue:
        level = []
        for _ in range(len(queue)):
            node = queue.popleft()
            level.append(node.val)
            queue.extend(c for c in (node.left, node.right) if c)
        out.append(level)
    return out


assert level_order(build_tree([3, 9, 20, None, None, 15, 7])) == [[3], [9, 20], [15, 7]]
assert level_order(None) == []
print(level_order(build_tree([3, 9, 20, None, None, 15, 7])))

### 3.1.2 [199. Binary Tree Right Side View](https://leetcode.com/problems/binary-tree-right-side-view/) — ★★ · Medium

**Problem.** Values visible from the right side (last node of each level).  
**Signal.** Level-order, keep the last value of each level.  
**Idea.** Run the level BFS and record `level[-1]`. (DFS variant: visit right first and record the first node seen at each new depth.)  
**Cost.** O(n) time · O(width) space

In [ ]:
def right_side_view(root: TreeNode | None) -> list[int]:
    """Rightmost value on each level."""
    return [level[-1] for level in level_order(root)]


assert right_side_view(build_tree([1, 2, 3, None, 5, None, 4])) == [1, 3, 4]
assert right_side_view(build_tree([1, 2, 3, 4, None, None, None, 5])) == [1, 3, 4, 5]
print(right_side_view(build_tree([1, 2, 3, None, 5, None, 4])))

### 3.1.3 [103. Binary Tree Zigzag Level Order Traversal](https://leetcode.com/problems/binary-tree-zigzag-level-order-traversal/) — ★★ · Medium

**Problem.** Level order, alternating left→right and right→left.  
**Signal.** Level BFS, reverse every other level.  
**Idea.** Same traversal; reverse odd-indexed levels before appending (or append to a deque from alternating ends).  
**Cost.** O(n) time

In [ ]:
def zigzag_level_order(root: TreeNode | None) -> list[list[int]]:
    """Level order with alternating direction."""
    levels = level_order(root)
    return [lvl if i % 2 == 0 else lvl[::-1] for i, lvl in enumerate(levels)]


assert zigzag_level_order(build_tree([3, 9, 20, None, None, 15, 7])) == [
    [3],
    [20, 9],
    [15, 7],
]
print(zigzag_level_order(build_tree([1, 2, 3, 4, None, None, 5])))

### 3.1.4 [637. Average of Levels in Binary Tree](https://leetcode.com/problems/average-of-levels-in-binary-tree/) — ★★ · Easy

**Problem.** Average value of each level.  
**Signal.** Level BFS.  
**Idea.** Average each level from the level-order output.  
**Cost.** O(n) time

In [ ]:
def average_of_levels(root: TreeNode | None) -> list[float]:
    """Mean value per level."""
    return [sum(lvl) / len(lvl) for lvl in level_order(root)]


assert average_of_levels(build_tree([3, 9, 20, None, None, 15, 7])) == [3.0, 14.5, 11.0]
print(average_of_levels(build_tree([3, 9, 20, None, None, 15, 7])))

### 3.1.5 [1161. Maximum Level Sum of a Binary Tree](https://leetcode.com/problems/maximum-level-sum-of-a-binary-tree/) — ★ · Medium

**Problem.** The smallest level number (root = 1) with the maximum sum.  
**Signal.** Level BFS.  
**Idea.** Sum each level; `max` over levels returns the first maximum on ties.  
**Cost.** O(n) time

In [ ]:
def max_level_sum(root: TreeNode | None) -> int:
    """1-based level with the largest sum."""
    sums = [sum(lvl) for lvl in level_order(root)]
    return sums.index(max(sums)) + 1


assert max_level_sum(build_tree([1, 7, 0, 7, -8, None, None])) == 2
assert (
    max_level_sum(
        build_tree([989, None, 10250, 98693, -89388, None, None, None, -32127])
    )
    == 2
)
print(max_level_sum(build_tree([1, 7, 0, 7, -8, None, None])))

### 3.1.6 [662. Maximum Width of Binary Tree](https://leetcode.com/problems/maximum-width-of-binary-tree/) — ★★ · Medium

**Problem.** Maximum width of any level, counting null gaps between the end nodes.  
**Signal.** Give nodes heap-style indices (`2i`, `2i + 1`); width = last − first + 1.  
**Idea.** BFS carrying `(node, index)`. Re-base indices per level (subtract the first index) so numbers stay small in deep, skewed trees.  
**Cost.** O(n) time · O(width) space

In [ ]:
def width_of_binary_tree(root: TreeNode | None) -> int:
    """Largest level width including gaps."""
    best = 0
    queue = deque([(root, 0)])
    while queue:
        first = queue[0][1]
        last = first
        for _ in range(len(queue)):
            node, idx = queue.popleft()
            idx -= first
            last = idx
            if node.left:
                queue.append((node.left, 2 * idx))
            if node.right:
                queue.append((node.right, 2 * idx + 1))
        best = max(best, last + 1)
    return best


assert width_of_binary_tree(build_tree([1, 3, 2, 5, 3, None, 9])) == 4
assert width_of_binary_tree(build_tree([1, 3, 2, 5, None, None, 9, 6, None, 7])) == 7
assert width_of_binary_tree(build_tree([1, 3, 2, 5])) == 2
print(width_of_binary_tree(build_tree([1, 3, 2, 5, 3, None, 9])))

### 3.1.7 [958. Check Completeness of a Binary Tree](https://leetcode.com/problems/check-completeness-of-a-binary-tree/) — ★ · Medium

**Problem.** Is the tree complete (all levels full except the last, filled from the left)?  
**Signal.** BFS including `None`s: no node may appear after the first `None`.  
**Idea.** Push children even when `None`. Once a `None` is popped, every later entry must also be `None`.  
**Cost.** O(n) time

In [ ]:
def is_complete_tree(root: TreeNode | None) -> bool:
    """True if the tree is complete."""
    queue = deque([root])
    seen_gap = False
    while queue:
        node = queue.popleft()
        if not node:
            seen_gap = True
        else:
            if seen_gap:
                return False
            queue.extend([node.left, node.right])
    return True


assert is_complete_tree(build_tree([1, 2, 3, 4, 5, 6]))
assert not is_complete_tree(build_tree([1, 2, 3, 4, 5, None, 7]))
print(is_complete_tree(build_tree([1, 2, 3, 4, 5, 6])))

### 3.1.8 [117. Populating Next Right Pointers in Each Node II](https://leetcode.com/problems/populating-next-right-pointers-in-each-node-ii/) — ★★ · Medium

**Problem.** Set each node's `next` to its right neighbour on the same level, O(1) extra space.  
**Signal.** Use the already-linked level above as a linked list to wire the level below.  
**Idea.** Walk the current level via `next` pointers, building the child level with a dummy head and a tail pointer. Then descend to `dummy.next`.  
**Cost.** O(n) time · O(1) space

In [ ]:
def connect(root: TreeNode | None) -> TreeNode | None:
    """Link each node to its right neighbour."""
    level = root
    while level:
        dummy = tail = TreeNode()
        node = level
        while node:
            for child in (node.left, node.right):
                if child:
                    tail.next = tail = child
            node = getattr(node, "next", None)
        tail.next = None
        level = dummy.next
    return root


t = connect(build_tree([1, 2, 3, 4, 5, None, 7]))
row, node = [], t.left.left
while node:
    row.append(node.val)
    node = node.next
assert row == [4, 5, 7] and t.left.next is t.right
print(row)

### 3.1.9 [314. Binary Tree Vertical Order Traversal](https://leetcode.com/problems/binary-tree-vertical-order-traversal/) — ★ · Medium · Premium

**Problem.** Group values by column (root at 0, left −1, right +1), top to bottom, left to right.  
**Signal.** BFS (for top-to-bottom order) carrying the column index.  
**Idea.** BFS with `(node, col)`; append values to `columns[col]`, then emit columns from min to max.  
**Cost.** O(n) time

In [ ]:
def vertical_order(root: TreeNode | None) -> list[list[int]]:
    """Columns of values, left to right."""
    if not root:
        return []
    columns: dict[int, list[int]] = defaultdict(list)
    queue = deque([(root, 0)])
    while queue:
        node, col = queue.popleft()
        columns[col].append(node.val)
        if node.left:
            queue.append((node.left, col - 1))
        if node.right:
            queue.append((node.right, col + 1))
    return [columns[c] for c in sorted(columns)]


assert vertical_order(build_tree([3, 9, 20, None, None, 15, 7])) == [
    [9],
    [3, 15],
    [20],
    [7],
]
assert vertical_order(build_tree([3, 9, 8, 4, 0, 1, 7])) == [
    [4],
    [9],
    [3, 0, 1],
    [8],
    [7],
]
print(vertical_order(build_tree([3, 9, 20, None, None, 15, 7])))

### 3.1.10 [987. Vertical Order Traversal of a Binary Tree](https://leetcode.com/problems/vertical-order-traversal-of-a-binary-tree/) — ★ · Hard

**Problem.** Same columns, but nodes sharing a row and column are ordered by value.  
**Signal.** Collect `(col, row, val)` triples and sort.  
**Idea.** DFS records each node's coordinates; sorting by `(col, row, val)` gives the required order; then group by column.  
**Cost.** O(n log n) time

In [ ]:
def vertical_traversal(root: TreeNode | None) -> list[list[int]]:
    """Vertical order with value tie-break."""
    cells: list[tuple[int, int, int]] = []

    def dfs(node: TreeNode | None, row: int, col: int) -> None:
        if node:
            cells.append((col, row, node.val))
            dfs(node.left, row + 1, col - 1)
            dfs(node.right, row + 1, col + 1)

    dfs(root, 0, 0)
    out: dict[int, list[int]] = defaultdict(list)
    for col, _, val in sorted(cells):
        out[col].append(val)
    return list(out.values())


assert vertical_traversal(build_tree([1, 2, 3, 4, 6, 5, 7])) == [
    [4],
    [2],
    [1, 5, 6],
    [3],
    [7],
]
print(vertical_traversal(build_tree([3, 9, 20, None, None, 15, 7])))

---
# 4) Binary Search Tree

BST invariant: every key in the left subtree < node < every key in the right subtree. Two
consequences carry most questions: **in-order traversal is sorted**, and a lookup only
ever goes one way at each node (O(h)).

## 4.1 BST properties and in-order traversal

### 4.1.1 [98. Validate Binary Search Tree](https://leetcode.com/problems/validate-binary-search-tree/) — ★★★ · Medium

**Problem.** Is the tree a valid BST (strict inequalities)?  
**Signal.** Each node must lie in an open interval inherited from its ancestors.  
**Idea.** Pass `(low, high)` down: the left child gets `(low, node.val)`, the right child gets `(node.val, high)`. Checking only parent–child pairs is the classic bug.  
**Cost.** O(n) time · O(h) stack

In [ ]:
def is_valid_bst(root: TreeNode | None) -> bool:
    """True if the tree satisfies the BST property."""

    def valid(node: TreeNode | None, low: float, high: float) -> bool:
        if not node:
            return True
        if not low < node.val < high:
            return False
        return valid(node.left, low, node.val) and valid(node.right, node.val, high)

    return valid(root, float("-inf"), float("inf"))


assert is_valid_bst(build_tree([2, 1, 3]))
assert not is_valid_bst(build_tree([5, 1, 4, None, None, 3, 6]))
assert not is_valid_bst(build_tree([5, 4, 6, None, None, 3, 7]))
print(is_valid_bst(build_tree([2, 1, 3])))

### 4.1.2 [230. Kth Smallest Element in a BST](https://leetcode.com/problems/kth-smallest-element-in-a-bst/) — ★★★ · Medium

**Problem.** k-th smallest key in a BST.  
**Signal.** In-order = sorted → stop at the k-th visit.  
**Idea.** Iterative in-order with a stack: go left as far as possible, pop, count, go right. Stop when the count reaches `k`.  
**Cost.** O(h + k) time · O(h) space

In [ ]:
def kth_smallest(root: TreeNode | None, k: int) -> int:
    """k-th smallest value in a BST."""
    stack: list[TreeNode] = []
    node = root
    while True:
        while node:
            stack.append(node)
            node = node.left
        node = stack.pop()
        k -= 1
        if k == 0:
            return node.val
        node = node.right


assert kth_smallest(build_tree([3, 1, 4, None, 2]), 1) == 1
assert kth_smallest(build_tree([5, 3, 6, 2, 4, None, None, 1]), 3) == 3
print(kth_smallest(build_tree([5, 3, 6, 2, 4, None, None, 1]), 3))

### 4.1.3 [235. Lowest Common Ancestor of a Binary Search Tree](https://leetcode.com/problems/lowest-common-ancestor-of-a-binary-search-tree/) — ★★★ · Medium

**Problem.** LCA of `p` and `q` in a BST.  
**Signal.** The LCA is the first node whose value lies between `p` and `q`.  
**Idea.** If both values are smaller than the node, go left; both larger, go right; otherwise this node splits them.  
**Cost.** O(h) time · O(1) space

In [ ]:
def lca_bst(root: TreeNode, p: TreeNode, q: TreeNode) -> TreeNode:
    """LCA in a BST."""
    node = root
    while True:
        if p.val < node.val and q.val < node.val:
            node = node.left
        elif p.val > node.val and q.val > node.val:
            node = node.right
        else:
            return node


t = build_tree([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5])
assert lca_bst(t, find(t, 2), find(t, 8)).val == 6
assert lca_bst(t, find(t, 2), find(t, 4)).val == 2
print(lca_bst(t, find(t, 3), find(t, 5)).val)

### 4.1.4 [530. Minimum Absolute Difference in BST](https://leetcode.com/problems/minimum-absolute-difference-in-bst/) — ★★ · Easy

**Problem.** Minimum difference between values of any two nodes.  
**Signal.** In sorted order, the closest pair is adjacent → in-order with a `prev` value.  
**Idea.** In-order traverse, comparing each value with the previous one.  
**Cost.** O(n) time · O(h) stack

In [ ]:
def get_minimum_difference(root: TreeNode | None) -> int:
    """Smallest gap between two BST values."""
    best, prev = float("inf"), None

    def inorder(node: TreeNode | None) -> None:
        nonlocal best, prev
        if not node:
            return
        inorder(node.left)
        if prev is not None:
            best = min(best, node.val - prev)
        prev = node.val
        inorder(node.right)

    inorder(root)
    return best


assert get_minimum_difference(build_tree([4, 2, 6, 1, 3])) == 1
assert get_minimum_difference(build_tree([1, 0, 48, None, None, 12, 49])) == 1
print(get_minimum_difference(build_tree([4, 2, 6, 1, 3])))

### 4.1.5 [173. Binary Search Tree Iterator](https://leetcode.com/problems/binary-search-tree-iterator/) — ★★ · Medium

**Problem.** Iterator over a BST in ascending order with O(1) amortised `next` and O(h) memory.  
**Signal.** Iterative in-order traversal, paused between calls.  
**Idea.** The stack holds the left spine. `next` pops a node and pushes the left spine of its right child. Each node is pushed and popped once.  
**Cost.** O(1) amortised per call · O(h) space

In [ ]:
class BSTIterator:
    """Lazy in-order iterator."""

    def __init__(self, root: TreeNode | None) -> None:
        self.stack: list[TreeNode] = []
        self._push_left(root)

    def _push_left(self, node: TreeNode | None) -> None:
        while node:
            self.stack.append(node)
            node = node.left

    def next(self) -> int:
        node = self.stack.pop()
        self._push_left(node.right)
        return node.val

    def has_next(self) -> bool:
        return bool(self.stack)


it = BSTIterator(build_tree([7, 3, 15, None, None, 9, 20]))
out = []
while it.has_next():
    out.append(it.next())
assert out == [3, 7, 9, 15, 20]
print(out)

### 4.1.6 [285. Inorder Successor in BST](https://leetcode.com/problems/inorder-successor-in-bst/) — ★★ · Medium · Premium

**Problem.** The node with the smallest key greater than `p.val`.  
**Signal.** BST descent remembering the last node where we went left.  
**Idea.** If `p.val < node.val`, this node is a candidate and the successor may be further left; otherwise go right.  
**Cost.** O(h) time · O(1) space

In [ ]:
def inorder_successor(root: TreeNode | None, p: TreeNode) -> TreeNode | None:
    """Next larger node in a BST."""
    succ, node = None, root
    while node:
        if p.val < node.val:
            succ, node = node, node.left
        else:
            node = node.right
    return succ


t = build_tree([5, 3, 6, 2, 4, None, None, 1])
assert inorder_successor(t, find(t, 4)).val == 5
assert inorder_successor(t, find(t, 6)) is None
print(inorder_successor(t, find(t, 1)).val)

### 4.1.7 [108. Convert Sorted Array to Binary Search Tree](https://leetcode.com/problems/convert-sorted-array-to-binary-search-tree/) — ★★ · Easy

**Problem.** Build a height-balanced BST from a sorted array.  
**Signal.** The middle element is the root; recurse on halves.  
**Idea.** Choosing the middle keeps subtree sizes within one, giving a balanced tree.  
**Cost.** O(n) time · O(log n) stack

In [ ]:
def sorted_array_to_bst(nums: list[int]) -> TreeNode | None:
    """Balanced BST from sorted values."""

    def make(lo: int, hi: int) -> TreeNode | None:
        if lo > hi:
            return None
        mid = (lo + hi) // 2
        return TreeNode(nums[mid], make(lo, mid - 1), make(mid + 1, hi))

    return make(0, len(nums) - 1)


t = sorted_array_to_bst([-10, -3, 0, 5, 9])
assert is_valid_bst(t) and is_balanced(t)
print(tree_to_list(t))

### 4.1.8 [701. Insert into a Binary Search Tree](https://leetcode.com/problems/insert-into-a-binary-search-tree/) — ★ · Medium

**Problem.** Insert a value into a BST (value not present); return the root.  
**Signal.** Walk down like a search; attach at the empty slot.  
**Idea.** Go left or right by comparison until the child in that direction is empty; place the new node there.  
**Cost.** O(h) time · O(1) space

In [ ]:
def insert_into_bst(root: TreeNode | None, val: int) -> TreeNode:
    """BST with val inserted."""
    if not root:
        return TreeNode(val)
    node = root
    while True:
        side = "left" if val < node.val else "right"
        child = getattr(node, side)
        if not child:
            setattr(node, side, TreeNode(val))
            return root
        node = child


t = insert_into_bst(build_tree([4, 2, 7, 1, 3]), 5)
assert tree_to_list(t) == [4, 2, 7, 1, 3, 5]
print(tree_to_list(t))

### 4.1.9 [450. Delete Node in a BST](https://leetcode.com/problems/delete-node-in-a-bst/) — ★★ · Medium

**Problem.** Delete the node with key `key` and return the root.  
**Signal.** Three cases: leaf, one child, two children (replace with in-order successor).  
**Idea.** Recurse to the node. With at most one child, return that child. With two children, copy the smallest value of the right subtree into the node and delete that value from the right subtree.  
**Cost.** O(h) time

In [ ]:
def delete_node(root: TreeNode | None, key: int) -> TreeNode | None:
    """BST with key removed."""
    if not root:
        return None
    if key < root.val:
        root.left = delete_node(root.left, key)
    elif key > root.val:
        root.right = delete_node(root.right, key)
    else:
        if not root.left or not root.right:
            return root.left or root.right
        succ = root.right
        while succ.left:
            succ = succ.left
        root.val = succ.val
        root.right = delete_node(root.right, succ.val)
    return root


t = delete_node(build_tree([5, 3, 6, 2, 4, None, 7]), 3)
assert tree_to_list(t) == [5, 4, 6, 2, None, None, 7]
assert tree_to_list(delete_node(build_tree([5, 3, 6, 2, 4, None, 7]), 0)) == [
    5,
    3,
    6,
    2,
    4,
    None,
    7,
]
print(tree_to_list(t))

### 4.1.10 [938. Range Sum of BST](https://leetcode.com/problems/range-sum-of-bst/) — ★ · Easy

**Problem.** Sum of values in `[low, high]`.  
**Signal.** Prune subtrees the BST property rules out.  
**Idea.** If the node is below `low`, only the right subtree can contribute; above `high`, only the left; otherwise add it and recurse on both sides.  
**Cost.** O(n) worst, faster with pruning

In [ ]:
def range_sum_bst(root: TreeNode | None, low: int, high: int) -> int:
    """Sum of BST values within [low, high]."""
    if not root:
        return 0
    if root.val < low:
        return range_sum_bst(root.right, low, high)
    if root.val > high:
        return range_sum_bst(root.left, low, high)
    return (
        root.val
        + range_sum_bst(root.left, low, high)
        + range_sum_bst(root.right, low, high)
    )


assert range_sum_bst(build_tree([10, 5, 15, 3, 7, None, 18]), 7, 15) == 32
assert range_sum_bst(build_tree([10, 5, 15, 3, 7, 13, 18, 1, None, 6]), 6, 10) == 23
print(range_sum_bst(build_tree([10, 5, 15, 3, 7, None, 18]), 7, 15))

### 4.1.11 [99. Recover Binary Search Tree](https://leetcode.com/problems/recover-binary-search-tree/) — ★ · Medium

**Problem.** Exactly two nodes were swapped; fix the BST without changing its structure.  
**Signal.** In-order sequence has one or two "drops" where `prev > cur`.  
**Idea.** In-order traverse. At the first drop, the swapped-in large value is `prev`; at the last drop, the swapped-in small value is `cur`. Swap the values of those two nodes.  
**Cost.** O(n) time · O(h) stack

In [ ]:
def recover_tree(root: TreeNode | None) -> None:
    """Swap back the two misplaced values in place."""
    first = second = prev = None

    def inorder(node: TreeNode | None) -> None:
        nonlocal first, second, prev
        if not node:
            return
        inorder(node.left)
        if prev and prev.val > node.val:
            if not first:
                first = prev
            second = node
        prev = node
        inorder(node.right)

    inorder(root)
    first.val, second.val = second.val, first.val


t = build_tree([1, 3, None, None, 2])
recover_tree(t)
assert tree_to_list(t) == [3, 1, None, None, 2]
t = build_tree([3, 1, 4, None, None, 2])
recover_tree(t)
assert tree_to_list(t) == [2, 1, 4, None, None, 3]
print(tree_to_list(t))

### 4.1.12 [426. Convert Binary Search Tree to Sorted Doubly Linked List](https://leetcode.com/problems/convert-binary-search-tree-to-sorted-doubly-linked-list/) — ★ · Medium · Premium

**Problem.** Convert a BST in place into a sorted *circular* doubly linked list (`left` = prev, `right` = next); return the smallest node.  
**Signal.** In-order traversal while linking each node to the previously visited one.  
**Idea.** Keep `first` and `last`. On visiting a node, link `last.right = node` and `node.left = last`. Finally connect `last` and `first` to close the circle.  
**Cost.** O(n) time · O(h) stack

In [ ]:
def tree_to_doubly_list(root: TreeNode | None) -> TreeNode | None:
    """Head of the circular sorted doubly linked list."""
    if not root:
        return None
    first = last = None

    def inorder(node: TreeNode | None) -> None:
        nonlocal first, last
        if not node:
            return
        inorder(node.left)
        if last:
            last.right, node.left = node, last
        else:
            first = node
        last = node
        inorder(node.right)

    inorder(root)
    last.right, first.left = first, last
    return first


head = tree_to_doubly_list(build_tree([4, 2, 5, 1, 3]))
vals, node = [], head
for _ in range(5):
    vals.append(node.val)
    node = node.right
assert vals == [1, 2, 3, 4, 5] and node is head and head.left.val == 5
print(vals)

---
# 5) Trie

A trie stores strings character by character; every node is a prefix. Lookups cost
O(length of the word), independent of how many words are stored, and all words sharing a
prefix live under one node — that is what makes autocomplete and multi-word grid search
fast.

## 5.1 Prefix trees

### 5.1.1 [208. Implement Trie (Prefix Tree)](https://leetcode.com/problems/implement-trie-prefix-tree/) — ★★★ · Medium

**Problem.** `insert`, `search` (whole word) and `starts_with` (prefix).  
**Signal.** Nested dicts, one level per character, with an end-of-word marker.  
**Idea.** Each node is a dict of children plus a flag `"$"` marking a complete word. `search` needs the flag; `starts_with` only needs the path to exist.  
**Cost.** O(L) per operation · O(total characters) space

In [ ]:
class Trie:
    """Prefix tree over lowercase words."""

    def __init__(self) -> None:
        self.root: dict = {}

    def insert(self, word: str) -> None:
        node = self.root
        for ch in word:
            node = node.setdefault(ch, {})
        node["$"] = True

    def _walk(self, s: str) -> dict | None:
        node = self.root
        for ch in s:
            if ch not in node:
                return None
            node = node[ch]
        return node

    def search(self, word: str) -> bool:
        node = self._walk(word)
        return bool(node and node.get("$"))

    def starts_with(self, prefix: str) -> bool:
        return self._walk(prefix) is not None


trie = Trie()
trie.insert("apple")
assert trie.search("apple") and not trie.search("app") and trie.starts_with("app")
trie.insert("app")
assert trie.search("app")
print("ok")

### 5.1.2 [211. Design Add and Search Words Data Structure](https://leetcode.com/problems/design-add-and-search-words-data-structure/) — ★★★ · Medium

**Problem.** `add_word(word)`; `search(pattern)` where `.` matches any letter.  
**Signal.** Trie + DFS that branches on `.`.  
**Idea.** On a normal character follow one child; on `.` try every child. Recursion depth is the word length.  
**Cost.** O(L) add · O(26^dots · L) worst-case search

In [ ]:
class WordDictionary:
    """Trie supporting '.' wildcards."""

    def __init__(self) -> None:
        self.root: dict = {}

    def add_word(self, word: str) -> None:
        node = self.root
        for ch in word:
            node = node.setdefault(ch, {})
        node["$"] = True

    def search(self, word: str) -> bool:
        def dfs(node: dict, i: int) -> bool:
            if i == len(word):
                return "$" in node
            if word[i] == ".":
                return any(
                    dfs(child, i + 1) for key, child in node.items() if key != "$"
                )
            return word[i] in node and dfs(node[word[i]], i + 1)

        return dfs(self.root, 0)


wd = WordDictionary()
for w in ["bad", "dad", "mad"]:
    wd.add_word(w)
assert (
    not wd.search("pad") and wd.search("bad") and wd.search(".ad") and wd.search("b..")
)
print("ok")

### 5.1.3 [212. Word Search II](https://leetcode.com/problems/word-search-ii/) — ★★★ · Hard

**Problem.** All dictionary words that can be traced in a letter grid (adjacent cells, no reuse).  
**Signal.** Many words over one grid → trie of words + one DFS per cell.  
**Idea.** Build a trie of all words. DFS from each cell, following trie edges only (a dead prefix stops the search immediately). Store the full word at its end node, record it on arrival and remove it to avoid duplicates. Pruning empty trie branches speeds things up.  
**Cost.** O(cells · 4 · 3^(L−1)) worst case

In [ ]:
def find_words(board: list[list[str]], words: list[str]) -> list[str]:
    """Dictionary words present in the grid."""
    root: dict = {}
    for w in words:
        node = root
        for ch in w:
            node = node.setdefault(ch, {})
        node["$"] = w
    rows, cols = len(board), len(board[0])
    found: list[str] = []

    def dfs(r: int, c: int, parent: dict) -> None:
        ch = board[r][c]
        node = parent[ch]
        if "$" in node:
            found.append(node.pop("$"))
        board[r][c] = "#"
        for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
            if 0 <= nr < rows and 0 <= nc < cols and board[nr][nc] in node:
                dfs(nr, nc, node)
        board[r][c] = ch
        if not node:
            parent.pop(ch)

    for r in range(rows):
        for c in range(cols):
            if board[r][c] in root:
                dfs(r, c, root)
    return found


board = [list("oaan"), list("etae"), list("ihkr"), list("iflv")]
assert sorted(find_words(board, ["oath", "pea", "eat", "rain"])) == ["eat", "oath"]
assert find_words([list("ab"), list("cd")], ["abcb"]) == []
print(find_words(board, ["oath", "pea", "eat", "rain"]))

### 5.1.4 [1268. Search Suggestions System](https://leetcode.com/problems/search-suggestions-system/) — ★ · Medium

**Problem.** After each typed character, return up to three lexicographically smallest products with that prefix.  
**Signal.** Sorted list + binary search on the prefix (or a trie storing top-3 per node).  
**Idea.** Sort products. For each prefix, `bisect_left` finds the first candidate; take the next three and keep those that start with the prefix.  
**Cost.** O(n log n + L log n) time

In [ ]:
def suggested_products(products: list[str], search_word: str) -> list[list[str]]:
    """Top-3 suggestions per typed prefix."""
    products = sorted(products)
    out, prefix = [], ""
    for ch in search_word:
        prefix += ch
        i = bisect_left(products, prefix)
        out.append([p for p in products[i : i + 3] if p.startswith(prefix)])
    return out


res = suggested_products(
    ["mobile", "mouse", "moneypot", "monitor", "mousepad"], "mouse"
)
assert res[0] == ["mobile", "moneypot", "monitor"]
assert res[-1] == ["mouse", "mousepad"]
print(res)

### 5.1.5 [648. Replace Words](https://leetcode.com/problems/replace-words/) — ★ · Medium

**Problem.** Replace every word in a sentence by its shortest dictionary root that is a prefix of it.  
**Signal.** Prefix lookup per word → trie, stop at the first end marker.  
**Idea.** Insert roots into a trie. For each word, walk the trie; return the prefix at the first end-of-word node, or the word itself if the walk fails.  
**Cost.** O(total characters)

In [ ]:
def replace_words(dictionary: list[str], sentence: str) -> str:
    """Sentence with words shortened to their roots."""
    root: dict = {}
    for w in dictionary:
        node = root
        for ch in w:
            node = node.setdefault(ch, {})
        node["$"] = True

    def shorten(word: str) -> str:
        node = root
        for i, ch in enumerate(word):
            if ch not in node:
                return word
            node = node[ch]
            if "$" in node:
                return word[: i + 1]
        return word

    return " ".join(shorten(w) for w in sentence.split())


out = replace_words(["cat", "bat", "rat"], "the cattle was rattled by the battery")
assert out == "the cat was rat by the bat"
print(out)

### 5.1.6 [421. Maximum XOR of Two Numbers in an Array](https://leetcode.com/problems/maximum-xor-of-two-numbers-in-an-array/) — ★ · Medium

**Problem.** Maximum `a XOR b` over pairs.  
**Signal.** Bitwise trie: greedily choose the opposite bit from the top.  
**Idea.** Insert numbers as 31-bit binary strings into a trie. For each number, walk from the most significant bit, preferring the child with the opposite bit (sets that XOR bit to 1). Higher bits dominate, so greedy is optimal.  
**Cost.** O(n · 31) time

In [ ]:
def find_maximum_xor(nums: list[int]) -> int:
    """Largest XOR of any pair."""
    root: dict = {}
    for x in nums:
        node = root
        for b in range(30, -1, -1):
            node = node.setdefault((x >> b) & 1, {})
    best = 0
    for x in nums:
        node, cur = root, 0
        for b in range(30, -1, -1):
            bit = (x >> b) & 1
            if 1 - bit in node:
                cur |= 1 << b
                node = node[1 - bit]
            else:
                node = node[bit]
        best = max(best, cur)
    return best


assert find_maximum_xor([3, 10, 5, 25, 2, 8]) == 28
assert find_maximum_xor([14, 70, 53, 83, 49, 91, 36, 80, 92, 51, 66, 70]) == 127
print(find_maximum_xor([3, 10, 5, 25, 2, 8]))

---
# 6) Heap / Priority Queue

| Goal | Heap | Size |
|---|---|---|
| k largest | **min**-heap, pop when size > k (the top is the k-th largest) | k |
| k smallest | **max**-heap (push negatives) | k |
| repeatedly take best | min/max heap of candidates | all candidates |
| median of a stream | max-heap for the lower half, min-heap for the upper half | n |
| merge k sorted sources | min-heap of the current head of each source | k |

## 6.1 Top-k and k-th element

### 6.1.1 [703. Kth Largest Element in a Stream](https://leetcode.com/problems/kth-largest-element-in-a-stream/) — ★★ · Easy

**Problem.** After each `add`, return the k-th largest element so far.  
**Signal.** k-th largest → min-heap of size k.  
**Idea.** Keep only the k largest values in a min-heap; its top is the k-th largest. Push, then pop if the heap grows past k.  
**Cost.** O(log k) per add · O(k) space

In [ ]:
class KthLargest:
    """Running k-th largest value."""

    def __init__(self, k: int, nums: list[int]) -> None:
        self.k = k
        self.heap: list[int] = []
        for x in nums:
            self.add(x)

    def add(self, val: int) -> int:
        heapq.heappush(self.heap, val)
        if len(self.heap) > self.k:
            heapq.heappop(self.heap)
        return self.heap[0]


kl = KthLargest(3, [4, 5, 8, 2])
assert [kl.add(v) for v in [3, 5, 10, 9, 4]] == [4, 5, 5, 8, 8]
print("ok")

### 6.1.2 [1046. Last Stone Weight](https://leetcode.com/problems/last-stone-weight/) — ★★ · Easy

**Problem.** Repeatedly smash the two heaviest stones; return the last weight (or 0).  
**Signal.** Repeatedly take the two largest → max-heap.  
**Idea.** Push negatives. Pop two; if they differ, push the difference back.  
**Cost.** O(n log n) time

In [ ]:
def last_stone_weight(stones: list[int]) -> int:
    """Weight of the final stone, or 0."""
    heap = [-s for s in stones]
    heapq.heapify(heap)
    while len(heap) > 1:
        a, b = -heapq.heappop(heap), -heapq.heappop(heap)
        if a != b:
            heapq.heappush(heap, -(a - b))
    return -heap[0] if heap else 0


assert last_stone_weight([2, 7, 4, 1, 8, 1]) == 1
assert last_stone_weight([1]) == 1
print(last_stone_weight([2, 7, 4, 1, 8, 1]))

### 6.1.3 [973. K Closest Points to Origin](https://leetcode.com/problems/k-closest-points-to-origin/) — ★★ · Medium

**Problem.** The `k` points closest to the origin.  
**Signal.** k smallest by distance → max-heap of size k (or `heapq.nsmallest`).  
**Idea.** Push `(-dist², point)`; pop when the size exceeds k, which removes the farthest of the kept points. Quickselect gives O(n) average.  
**Cost.** O(n log k) time · O(k) space

In [ ]:
def k_closest(points: list[list[int]], k: int) -> list[list[int]]:
    """k points nearest the origin."""
    heap: list[tuple[int, list[int]]] = []
    for x, y in points:
        heapq.heappush(heap, (-(x * x + y * y), [x, y]))
        if len(heap) > k:
            heapq.heappop(heap)
    return [p for _, p in heap]


assert k_closest([[1, 3], [-2, 2]], 1) == [[-2, 2]]
assert sorted(k_closest([[3, 3], [5, -1], [-2, 4]], 2)) == [[-2, 4], [3, 3]]
print(k_closest([[3, 3], [5, -1], [-2, 4]], 2))

### 6.1.4 [215. Kth Largest Element in an Array](https://leetcode.com/problems/kth-largest-element-in-an-array/) — ★★ · Medium

**Problem.** k-th largest element (not distinct), ideally better than sorting.  
**Signal.** Min-heap of size k (O(n log k)) or Quickselect (O(n) average).  
**Idea.** Quickselect: partition around a random pivot into greater / equal / less. Recurse only into the part that contains the k-th largest. The three-way split handles duplicates.  
**Cost.** O(n) average, O(n²) worst · O(n) space here

In [ ]:
def find_kth_largest(nums: list[int], k: int) -> int:
    """k-th largest value via quickselect."""
    pivot = random.choice(nums)
    greater = [x for x in nums if x > pivot]
    equal = [x for x in nums if x == pivot]
    if k <= len(greater):
        return find_kth_largest(greater, k)
    if k <= len(greater) + len(equal):
        return pivot
    less = [x for x in nums if x < pivot]
    return find_kth_largest(less, k - len(greater) - len(equal))


assert find_kth_largest([3, 2, 1, 5, 6, 4], 2) == 5
assert find_kth_largest([3, 2, 3, 1, 2, 4, 5, 5, 6], 4) == 4
print(find_kth_largest([3, 2, 1, 5, 6, 4], 2))

### 6.1.5 [692. Top K Frequent Words](https://leetcode.com/problems/top-k-frequent-words/) — ★★ · Medium

**Problem.** `k` most frequent words; ties broken alphabetically.  
**Signal.** Top k with a composite key `(−count, word)`.  
**Idea.** Count, then `heapq.nsmallest(k, …, key=(−count, word))` — the key orders higher counts first and alphabetical order on ties.  
**Cost.** O(n log k) time

In [ ]:
def top_k_frequent_words(words: list[str], k: int) -> list[str]:
    """k most frequent words, alphabetical on ties."""
    count = Counter(words)
    return heapq.nsmallest(k, count, key=lambda w: (-count[w], w))


assert top_k_frequent_words(["i", "love", "leetcode", "i", "love", "coding"], 2) == [
    "i",
    "love",
]
words = ["the", "day", "is", "sunny", "the", "the", "the", "sunny", "is", "is"]
assert top_k_frequent_words(words, 4) == ["the", "is", "sunny", "day"]
print(top_k_frequent_words(words, 4))

### 6.1.6 [373. Find K Pairs with Smallest Sums](https://leetcode.com/problems/find-k-pairs-with-smallest-sums/) — ★★ · Medium

**Problem.** Sorted arrays `a`, `b`; the `k` pairs `(a[i], b[j])` with the smallest sums.  
**Signal.** Merge k sorted lists: row `i` is `a[i] + b[0], a[i] + b[1], …`.  
**Idea.** Seed the heap with `(a[i] + b[0], i, 0)` for the first `k` rows. Pop the smallest and push its right neighbour `(i, j + 1)`. Only k pops are needed.  
**Cost.** O(k log k) time

In [ ]:
def k_smallest_pairs(a: list[int], b: list[int], k: int) -> list[list[int]]:
    """k pairs with the smallest sums."""
    heap = [(a[i] + b[0], i, 0) for i in range(min(k, len(a)))]
    out = []
    while heap and len(out) < k:
        _, i, j = heapq.heappop(heap)
        out.append([a[i], b[j]])
        if j + 1 < len(b):
            heapq.heappush(heap, (a[i] + b[j + 1], i, j + 1))
    return out


assert k_smallest_pairs([1, 7, 11], [2, 4, 6], 3) == [[1, 2], [1, 4], [1, 6]]
assert k_smallest_pairs([1, 1, 2], [1, 2, 3], 2) == [[1, 1], [1, 1]]
print(k_smallest_pairs([1, 7, 11], [2, 4, 6], 3))

### 6.1.7 [264. Ugly Number II](https://leetcode.com/problems/ugly-number-ii/) — ★ · Medium

**Problem.** The n-th number whose only prime factors are 2, 3 and 5.  
**Signal.** Merge three sorted streams (`2×ugly`, `3×ugly`, `5×ugly`) with three pointers.  
**Idea.** Each ugly number is a smaller ugly number times 2, 3 or 5. Keep a pointer per factor; the next ugly number is the min of the three candidates; advance every pointer that produced it (this dedupes, e.g. 6 = 2×3 = 3×2).  
**Cost.** O(n) time · O(n) space

In [ ]:
def nth_ugly_number(n: int) -> int:
    """n-th 2-3-5 smooth number."""
    ugly = [1]
    i2 = i3 = i5 = 0
    while len(ugly) < n:
        nxt = min(ugly[i2] * 2, ugly[i3] * 3, ugly[i5] * 5)
        ugly.append(nxt)
        i2 += nxt == ugly[i2] * 2
        i3 += nxt == ugly[i3] * 3
        i5 += nxt == ugly[i5] * 5
    return ugly[-1]


assert nth_ugly_number(10) == 12
assert nth_ugly_number(1) == 1
print(nth_ugly_number(10))

## 6.2 Scheduling and simulation with heaps

### 6.2.1 [621. Task Scheduler](https://leetcode.com/problems/task-scheduler/) — ★★ · Medium

**Problem.** Tasks with a cooldown `n` between equal tasks; minimum time slots (idle allowed).  
**Signal.** The most frequent task dictates the frame → counting formula.  
**Idea.** Let `max_count` be the highest frequency and `ties` the number of tasks with that frequency. The frame `(max_count − 1) × (n + 1) + ties` is a lower bound; if there are enough other tasks to fill every idle slot, the answer is simply `len(tasks)`.  
**Cost.** O(N) time · O(26) space

In [ ]:
def least_interval(tasks: list[str], n: int) -> int:
    """Fewest slots to run all tasks with cooldown n."""
    counts = Counter(tasks).values()
    max_count = max(counts)
    ties = sum(1 for c in counts if c == max_count)
    return max(len(tasks), (max_count - 1) * (n + 1) + ties)


assert least_interval(list("AAABBB"), 2) == 8
assert least_interval(list("ACABDB"), 1) == 6
assert least_interval(list("AAABBB"), 3) == 10
print(least_interval(list("AAABBB"), 2))

### 6.2.2 [767. Reorganize String](https://leetcode.com/problems/reorganize-string/) — ★ · Medium

**Problem.** Rearrange so no two adjacent characters are equal, or return `""`.  
**Signal.** Always place the most frequent remaining character that isn't the previous one → max-heap.  
**Idea.** Pop the top character, place it, and hold it out for one step (push back the previous one). Impossible iff some count exceeds `(n + 1) // 2`.  
**Cost.** O(n log 26) time

In [ ]:
def reorganize_string(s: str) -> str:
    """Arrangement with no equal neighbours, or empty."""
    heap = [(-c, ch) for ch, c in Counter(s).items()]
    heapq.heapify(heap)
    out: list[str] = []
    prev = None
    while heap:
        c, ch = heapq.heappop(heap)
        out.append(ch)
        if prev:
            heapq.heappush(heap, prev)
        prev = (c + 1, ch) if c + 1 < 0 else None
    return "".join(out) if len(out) == len(s) else ""


res = reorganize_string("aab")
assert res == "aba"
assert reorganize_string("aaab") == ""
assert all(
    a != b for a, b in zip(reorganize_string("aaabbc"), reorganize_string("aaabbc")[1:])
)
print(res)

### 6.2.3 [355. Design Twitter](https://leetcode.com/problems/design-twitter/) — ★★ · Medium

**Problem.** `post_tweet`, `follow`, `unfollow`, and `get_news_feed` (10 most recent tweets from the user and followees).  
**Signal.** Merge k sorted tweet lists → heap over each followee's latest tweet.  
**Idea.** Store tweets per user as `(time, id)` with a global decreasing timestamp. For the feed, seed a heap with each followee's newest tweet and pop up to 10 times, pushing that user's next older tweet.  
**Cost.** O(F + 10 log F) per feed for F followees

In [ ]:
class Twitter:
    """Minimal Twitter feed."""

    def __init__(self) -> None:
        self.time = 0
        self.tweets: dict[int, list[tuple[int, int]]] = defaultdict(list)
        self.following: dict[int, set[int]] = defaultdict(set)

    def post_tweet(self, user_id: int, tweet_id: int) -> None:
        self.time -= 1
        self.tweets[user_id].append((self.time, tweet_id))

    def get_news_feed(self, user_id: int) -> list[int]:
        heap = []
        for u in self.following[user_id] | {user_id}:
            if self.tweets[u]:
                i = len(self.tweets[u]) - 1
                heap.append((self.tweets[u][i][0], u, i))
        heapq.heapify(heap)
        feed = []
        while heap and len(feed) < 10:
            _, u, i = heapq.heappop(heap)
            feed.append(self.tweets[u][i][1])
            if i:
                heapq.heappush(heap, (self.tweets[u][i - 1][0], u, i - 1))
        return feed

    def follow(self, follower: int, followee: int) -> None:
        if follower != followee:
            self.following[follower].add(followee)

    def unfollow(self, follower: int, followee: int) -> None:
        self.following[follower].discard(followee)


tw = Twitter()
tw.post_tweet(1, 5)
assert tw.get_news_feed(1) == [5]
tw.follow(1, 2)
tw.post_tweet(2, 6)
assert tw.get_news_feed(1) == [6, 5]
tw.unfollow(1, 2)
assert tw.get_news_feed(1) == [5]
print("ok")

### 6.2.4 [295. Find Median from Data Stream](https://leetcode.com/problems/find-median-from-data-stream/) — ★★★ · Hard

**Problem.** `add_num` and `find_median` over a stream.  
**Signal.** Two heaps: max-heap for the lower half, min-heap for the upper half.  
**Idea.** Push to the lower half, then move its largest to the upper half (keeps order). If the upper half becomes larger, move its smallest back. The lower half holds the extra element for odd counts, so the median is its top, or the average of both tops.  
**Cost.** O(log n) add · O(1) median

In [ ]:
class MedianFinder:
    """Streaming median with two heaps."""

    def __init__(self) -> None:
        self.low: list[int] = []
        self.high: list[int] = []

    def add_num(self, num: int) -> None:
        heapq.heappush(self.low, -num)
        heapq.heappush(self.high, -heapq.heappop(self.low))
        if len(self.high) > len(self.low):
            heapq.heappush(self.low, -heapq.heappop(self.high))

    def find_median(self) -> float:
        if len(self.low) > len(self.high):
            return float(-self.low[0])
        return (-self.low[0] + self.high[0]) / 2


mf = MedianFinder()
mf.add_num(1)
mf.add_num(2)
assert mf.find_median() == 1.5
mf.add_num(3)
assert mf.find_median() == 2.0
print(mf.find_median())

### 6.2.5 [502. IPO](https://leetcode.com/problems/ipo/) — ★★ · Hard

**Problem.** Start with capital `w`; do at most `k` projects, each needs `capital[i]` and yields `profits[i]`. Maximise final capital.  
**Signal.** Greedy: among affordable projects, always take the most profitable → sort by capital + max-heap of profits.  
**Idea.** Sort projects by required capital. Each round, push every newly affordable project's profit onto a max-heap, then take the best one. Capital only grows, so a project once affordable stays affordable.  
**Cost.** O(n log n + k log n)

In [ ]:
def find_maximized_capital(
    k: int, w: int, profits: list[int], capital: list[int]
) -> int:
    """Capital after at most k greedy projects."""
    projects = sorted(zip(capital, profits))
    heap: list[int] = []
    i = 0
    for _ in range(k):
        while i < len(projects) and projects[i][0] <= w:
            heapq.heappush(heap, -projects[i][1])
            i += 1
        if not heap:
            break
        w -= heapq.heappop(heap)
    return w


assert find_maximized_capital(2, 0, [1, 2, 3], [0, 1, 1]) == 4
assert find_maximized_capital(3, 0, [1, 2, 3], [0, 1, 2]) == 6
print(find_maximized_capital(2, 0, [1, 2, 3], [0, 1, 1]))

### 6.2.6 [1834. Single-Threaded CPU](https://leetcode.com/problems/single-threaded-cpu/) — ★ · Medium

**Problem.** Tasks `[enqueue_time, processing_time]`; the CPU always picks the available task with the shortest processing time (ties by index). Return the processing order.  
**Signal.** Event simulation: sort by enqueue time + min-heap of available tasks.  
**Idea.** Advance time; push every task whose enqueue time ≤ time onto a heap keyed by `(processing, index)`. If the heap is empty, jump time to the next enqueue time. Pop and run one task, adding its processing time.  
**Cost.** O(n log n)

In [ ]:
def get_order(tasks: list[list[int]]) -> list[int]:
    """Indices in the order the CPU processes them."""
    order = sorted(range(len(tasks)), key=lambda i: tasks[i][0])
    heap: list[tuple[int, int]] = []
    out, time, i = [], 0, 0
    while len(out) < len(tasks):
        if not heap and time < tasks[order[i]][0]:
            time = tasks[order[i]][0]
        while i < len(order) and tasks[order[i]][0] <= time:
            heapq.heappush(heap, (tasks[order[i]][1], order[i]))
            i += 1
        duration, idx = heapq.heappop(heap)
        time += duration
        out.append(idx)
    return out


assert get_order([[1, 2], [2, 4], [3, 2], [4, 1]]) == [0, 2, 3, 1]
assert get_order([[7, 10], [7, 12], [7, 5], [7, 4], [7, 2]]) == [4, 3, 2, 0, 1]
print(get_order([[1, 2], [2, 4], [3, 2], [4, 1]]))

### 6.2.7 [1642. Furthest Building You Can Reach](https://leetcode.com/problems/furthest-building-you-can-reach/) — ★ · Medium

**Problem.** Climbing up needs bricks (height difference) or one ladder. How far can you go?  
**Signal.** Use ladders on the largest climbs → min-heap of the climbs currently given ladders.  
**Idea.** Tentatively give every climb a ladder (push onto a min-heap). When ladders run out, the smallest laddered climb is paid in bricks instead. If bricks go negative, stop.  
**Cost.** O(n log L)

In [ ]:
def furthest_building(heights: list[int], bricks: int, ladders: int) -> int:
    """Furthest reachable index."""
    heap: list[int] = []
    for i in range(len(heights) - 1):
        climb = heights[i + 1] - heights[i]
        if climb <= 0:
            continue
        heapq.heappush(heap, climb)
        if len(heap) > ladders:
            bricks -= heapq.heappop(heap)
            if bricks < 0:
                return i
    return len(heights) - 1


assert furthest_building([4, 2, 7, 6, 9, 14, 12], 5, 1) == 4
assert furthest_building([4, 12, 2, 7, 3, 18, 20, 3, 19], 10, 2) == 7
assert furthest_building([14, 3, 19, 3], 17, 0) == 3
print(furthest_building([4, 2, 7, 6, 9, 14, 12], 5, 1))

### 6.2.8 [632. Smallest Range Covering Elements from K Lists](https://leetcode.com/problems/smallest-range-covering-elements-from-k-lists/) — ★★ · Hard

**Problem.** Smallest range `[a, b]` containing at least one number from each of `k` sorted lists.  
**Signal.** Sliding a window across k sorted lists → min-heap of current heads + running max.  
**Idea.** Keep one pointer per list in a min-heap, plus the max of the current heads. The range `[heap_min, current_max]` covers all lists; record it, then advance the list holding the minimum (the only way to shrink the range). Stop when any list is exhausted.  
**Cost.** O(N log k) for N total numbers

In [ ]:
def smallest_range(nums: list[list[int]]) -> list[int]:
    """Tightest range touching every list."""
    heap = [(row[0], i, 0) for i, row in enumerate(nums)]
    heapq.heapify(heap)
    hi = max(row[0] for row in nums)
    best = [heap[0][0], hi]
    while True:
        lo, i, j = heapq.heappop(heap)
        if hi - lo < best[1] - best[0]:
            best = [lo, hi]
        if j + 1 == len(nums[i]):
            return best
        nxt = nums[i][j + 1]
        hi = max(hi, nxt)
        heapq.heappush(heap, (nxt, i, j + 1))


lists = [[4, 10, 15, 24, 26], [0, 9, 12, 20], [5, 18, 22, 30]]
assert smallest_range(lists) == [20, 24]
assert smallest_range([[1, 2, 3], [1, 2, 3], [1, 2, 3]]) == [1, 1]
print(smallest_range(lists))

---
# 7) Conclusion

| Pattern | Recognise it by | Core move | Cost |
|---|---|---|---|
| Bottom-up DFS | height, diameter, balanced, max path, LCA | combine child results, update global best | O(n) |
| Tree DP (state pair) | rob/skip, cameras, coins | return a tuple of states per node | O(n) |
| Top-down DFS | path sums, good nodes, bounds | pass state down, backtrack shared lists | O(n) |
| Prefix sum on paths | downward paths summing to k | hash map of prefix sums, undo on return | O(n) |
| Construction | pre/in, in/post, serialise | root from one order, split by the other | O(n) |
| BFS by level | level lists, right view, width, completeness | `for _ in range(len(queue))` | O(n) |
| BST bounds / in-order | validate, k-th, successor, recover | `(low, high)` window; in-order is sorted | O(h)–O(n) |
| Trie | prefix queries, many words on one grid, max XOR | nested dicts + end marker | O(L) |
| Heap size k | top-k, k-th, k closest | min-heap for largest, max-heap for smallest | O(n log k) |
| Two heaps | streaming median | lower max-heap + upper min-heap | O(log n) |
| k-way merge | merge lists, smallest pairs, smallest range, feed | heap of current heads | O(N log k) |
| Greedy + heap | IPO, ladders, CPU scheduling, reorganise | sort by availability, heap by value | O(n log n) |